# Video-only xG end-to-end experiment

Notebook uruchamia produkcyjny runner end-to-end i czyta artefakty zapisane na dysku. Logika eksperymentu znajduje się w `tactifoot_vision.video_xg`, a StatsBomb jest ładowany wyłącznie w etapie ewaluacji runnera.

In [ ]:
from pathlib import Path
import json
import os
import sys
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = Path('/home/kuba/projects/tactifoot_vision').resolve()
os.chdir(REPO_ROOT)
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from tactifoot_vision.video_xg import (
    VideoOnlyXgEndToEndRunner,
    load_video_only_xg_end_to_end_config,
)

CONFIG_PATH = REPO_ROOT / 'configs/experiments/video_xg_end_to_end_fa_wsl.yaml'
config = load_video_only_xg_end_to_end_config(CONFIG_PATH)
RUN_DIR = config.output_dir

print(f'CONFIG_PATH={CONFIG_PATH}')
print(f'RUN_DIR={RUN_DIR}')

## End-to-end full video run

Ustaw `RUN_END_TO_END=True`, żeby uruchomić pełny pipeline na `part1.mp4` i `part2.mp4`. `STOP_AFTER`, `RESUME_FROM` i `FORCE_STAGE` odpowiadają flagom CLI.

In [ ]:
RUN_END_TO_END = False
RESUME_FROM = None
STOP_AFTER = None
FORCE_STAGE = None

if RUN_END_TO_END:
    result = VideoOnlyXgEndToEndRunner().run(
        config,
        resume_from=RESUME_FROM,
        stop_after=STOP_AFTER,
        force_stage=FORCE_STAGE,
    )
    print(json.dumps({
        'output_dir': str(result.output_dir),
        'artifacts': len(result.artifacts),
        'metrics': result.metrics,
    }, indent=2))
else:
    print('RUN_END_TO_END=False; czytam istniejące artefakty, jeżeli są dostępne.')

## Artefakty runu

In [ ]:
expected_artifacts = [
    '00_video_timeline.json',
    '01_sampled_frames.parquet',
    '02_detections.parquet',
    '03_tracks.parquet',
    '04_homographies.parquet',
    '04_projection_quality.csv',
    '05_ball_trajectory.parquet',
    '06_shot_candidates.parquet',
    '07_refined_shots.parquet',
    '08_video_features.csv',
    '08_video_features_model.csv',
    '09_predictions.csv',
    '10_per_shot_eval.csv',
    '10_method_metrics.csv',
    '10_metrics.json',
    'final_report.md',
]

artifact_status = pd.DataFrame({
    'artifact': expected_artifacts,
    'exists': [(RUN_DIR / name).exists() for name in expected_artifacts],
    'path': [str(RUN_DIR / name) for name in expected_artifacts],
})
display(artifact_status)

## Predykcje i metryki

In [ ]:
predictions_path = RUN_DIR / '09_predictions.csv'
method_metrics_path = RUN_DIR / '10_method_metrics.csv'
metrics_path = RUN_DIR / '10_metrics.json'
report_path = RUN_DIR / 'final_report.md'

if predictions_path.exists():
    predictions = pd.read_csv(predictions_path)
    display(predictions.head())
    display(predictions.groupby('method', as_index=False)['xg'].agg(['count', 'sum', 'mean']))
else:
    print(f'Brak predykcji: {predictions_path}')

if method_metrics_path.exists():
    display(pd.read_csv(method_metrics_path))

if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text()), indent=2))

if report_path.exists():
    print(report_path.read_text())